## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Planning a retention study: log-rank test sample size

**Sample Size and Power Calculation** · a worked example from the StatNex teaching guide

**The story.** Before running a study you must decide how many people to include. A company plans a study of how long customers stay before they cancel: does a loyalty programme lift 5-year retention from 50% to 65%? The calculation uses no data, only its settings.

**The question.** How many customers per group does the retention study need so that a real effect is unlikely to be missed?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `13_sample_size/sample_size__effect_sizes.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Test,Measure,Small,Medium,Large,Notes
Difference test (t test),r,0.1,0.3,0.5,The same values apply to the paired t test and the two-sample t test. Two measures are available: r and d.
,d,0.2,0.5,0.8,
Analysis of variance,eta squared,0.01,0.06,0.14,Multiple-comparison methods use r from the difference test (t test)
Correlation,r,0.1,0.3,0.5,The correlation coefficient itself
Chi-square test (2x2 table),phi,0.1,0.3,0.5,The association coefficient
Chi-square test (other tables),Cramer's V,0.1,0.3,0.5,The association coefficient
Difference test (nonparametric test),r,0.1,0.3,0.5,"For the Mann-Whitney, Wilcoxon, Kruskal-Wallis and Friedman tests, get Z from Z = r * sqrt(n)"
Multiple regression,R squared,0.02,0.13,0.26,The coefficient of determination
""")

## Libraries

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import optimize, stats
from statsmodels.stats.power import TTestIndPower, TTestPower

pd.set_option("display.width", 120)

# No data file: everything comes from the numbers set below.
alpha = 5 / 100
sides = ["two-sided", "one-sided"]


def z(p):
    """The standard normal value with probability p above it."""
    return stats.norm.isf(p)

## Events and sample size for a log-rank test

In [ ]:
# From the two groups' survival probabilities at the end of follow-up. Both methods
# first count the events (deaths) needed, then divide by the share of people expected
# to have one. Always two-sided.
power = 80 / 100
survival1, survival2 = 0.65, 0.5
hazard_ratio = math.log(survival1) / math.log(survival2)
z_sum = z(alpha / 2) + z(1 - power)
freedman = ((1 + hazard_ratio) / (1 - hazard_ratio)) ** 2 * z_sum**2 / 2
schoenfeld = 2 * z_sum**2 / math.log(hazard_ratio) ** 2
event_share = 1 - (survival1 + survival2) / 2
survival = pd.DataFrame(
    {"Freedman": [freedman, freedman / event_share, math.ceil(freedman / event_share)],
     "Schoenfeld": [schoenfeld, schoenfeld / event_share, math.ceil(schoenfeld / event_share)]},
    index=["events per group", "n per group", "rounded up"],
)
print(survival)
print(f"\nhazard ratio = {hazard_ratio:.6g}")

## Benchmarks for effect sizes (Cohen)

In [ ]:
# Rules of thumb for when nothing better is known: plan from the smallest effect that
# matters to the decision instead. The Cramer's V row repeats Cohen's values for w,
# which hold for V only when the smaller side of the table has two categories; for a
# larger table divide them by sqrt(min(rows, columns) - 1).
benchmarks = pd.DataFrame(
    [["t test", "r", 0.1, 0.3, 0.5], ["t test", "d", 0.2, 0.5, 0.8],
     ["ANOVA", "eta^2", 0.01, 0.06, 0.14], ["correlation", "r", 0.1, 0.3, 0.5],
     ["chi-square (2x2)", "phi", 0.1, 0.3, 0.5], ["chi-square (other)", "Cramer's V", 0.1, 0.3, 0.5],
     ["non-parametric", "r", 0.1, 0.3, 0.5], ["multiple regression", "R^2", 0.02, 0.13, 0.26]],
    columns=["analysis", "index", "small", "medium", "large"],
)
print()
print(benchmarks.to_string(index=False))

## What the result says

The log-rank test compares when customers cancel in two groups. The plan expects 65% of one group and 50% of the other still to be customers after 5 years. It sets a 5% significance level (the accepted risk of a false alarm) and 80% power (the chance of detecting the effect if it is real).

The hazard ratio is 0.62: one group would cancel at 62% of the other group's rate at any moment. The calculation first asks for events. The output calls them deaths, a word from medical studies; here an event is a cancellation. Freedman's method needs 72 per group, which takes 170 customers per group. Schoenfeld's method gives 164.

So plan for about 170 customers per group. The cancellations drive the power, and the answer is only as good as the effect you plan for.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.